# Experiment 3: Intermediate target near a 10-point validation accuracy gap

> **Previous outputs retained:** the displayed outputs below are from the old
> fixed-30-epoch run (19.94-point train-test gap), not the new 10-point
> validation-gap rule. The updated experiment has not been run. Restart the
> kernel and rerun in order when authorized; do not combine old and new results.

This is a target-training comparison with
`exp_3_expanded_missing_feature_baseline_mlp.ipynb` (the fixed 100-epoch reference).
The target uses ACS 2023; auxiliary records come from ACS 2014.
Only the target stopping rule changes. Keep the same 48 inputs, seed, target
partitions, preprocessing, architecture (512, 256, 128), optimizer and learning rate.

The 20-shadow reference pool stays fixed: six random forests, five gradient
boosting models, five XGBoost models and four 100-epoch MLPs. The shadow MLPs
retain the reference training recipe, independently of this target's stopping rule.
This holds the auxiliary attack construction fixed across target conditions.
Shadows are rebuilt deterministically in each notebook; no saved models are loaded.

Run the complete-feature MIA, seven-mask PFV baseline and K=6 conditional
single/average/mean-only/uncertainty comparisons in the same order as the reference.
The original blind-walk protocol, reconstruction and calibration procedures stay fixed.
Target queries are black box; the study assumes knowledge of the target architecture,
training setup and true income class for the attack's loss feature.

The intended regimes are better-generalized (early stopping), intermediate
(approximately 10 percentage points of train-validation accuracy gap,
within at most 30 epochs) and the existing 100-epoch reference. These are hypotheses about
training behavior: report the measured train/validation/test losses and accuracy
gaps before describing a target as well-generalized or moderately overfitted.
Validation-gap selection targets an exploratory overfitting condition. It does
not guarantee an exact 10-point gap, a 10-point test gap or a particular MIA
accuracy. Never choose an epoch using target test or membership results.

New outputs go to a separate gap10 artifact directory. The old fixed-30-epoch
artifacts and the 100-epoch reference notebook remain unchanged. No disk checkpoints or
logs are used. Running the full uncertainty section can still be expensive.
Run experiments only after explicit user approval.


In [ ]:
from pathlib import Path
import json
import time
import importlib
import src.expanded_data as expanded_data_module
import src.shadow_models as shadow_models_module
importlib.reload(expanded_data_module)
importlib.reload(shadow_models_module)
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from src.data import load_acs_income, split_acs_data, subset_target_training
from src.expanded_data import (
    load_expanded_acs, preprocess_expanded_partitions, expanded_feature_inventory,
    reference_split_config, EXPANDED_FEATURES, EXCLUDED, load_expanded_year,
)
import src.training as training_module
importlib.reload(training_module)
from src.training import train_target_model, evaluate_model
from src.shadow_models import (load_adult_data, train_shadow_model, train_shadow_models,
                               summarize_shadow_models, PAPER_SHADOW_ALGORITHMS)
from src.membership_attack import train_membership_classifier, evaluate_attack
from src.paper_attack_features import (collect_shadow_paper_features, collect_target_paper_features,
                                       make_probability_oracle)

torch.set_num_threads(4)
SEED = 42
TRAIN_SIZE = 100_000
INCOME_BASE_YEAR = None
NOMINAL_INCOME_METADATA = dict(enabled=False, label='PINCP > 50000', adjustment='None')
EXPERIMENT_DIR = Path('artifacts/exp_3_2023_2014_100000_nominal_tree16_mlp4_target_gap10_max30')
RUN_DIR = EXPERIMENT_DIR / 'reduced_k6_s20_m7'
RUN_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_CONFIG = reference_split_config()
assert SPLIT_CONFIG['seed'] == SEED, 'Use the reference seed consistently.'
TARGET_REGIME = 'intermediate_gap10'
TRAIN_CONFIG = dict(hidden_dims=(512, 256, 128), dropout=0.0, epochs=30,
    batch_size=256, learning_rate=1e-3, weight_decay=0.0, seed=SEED,
    device='cpu', early_stopping=False, restore_best=False,
    patience=5, monitor_train=True, target_gap_pp=10.0)
TARGET_TRAINING_PROTOCOL = dict(
    regime=TARGET_REGIME, max_epochs=TRAIN_CONFIG['epochs'],
    selection='stop at first train-validation accuracy-gap crossing; restore closest visited epoch',
    target_gap_pp=TRAIN_CONFIG['target_gap_pp'], tie_break='earlier epoch',
    if_target_not_reached='return closest visited epoch at 30-epoch cap',
    target_test_used_for_selection=False,
    target_mia_used_for_selection=False,
    shadow_schedule='fixed 100-epoch reference recipe',
    reference_notebook='exp_3_expanded_missing_feature_baseline_mlp.ipynb')
WALK_CONFIG = dict(max_steps=20, repeats=2, categorical_change_probability=0.05,
                   batch_size=512, seed=SEED)
SHADOW_RECORDS_PER_CLASS = 2000
N_SHADOW_MODELS = 20
SHADOW_ALGORITHMS = PAPER_SHADOW_ALGORITHMS + ('mlp',)
# Freeze the original shadow recipe; do not inherit target stopping settings.
SHADOW_MLP_CONFIG = dict(hidden_dims=(512, 256, 128), dropout=0.0, epochs=100,
    batch_size=256, learning_rate=1e-3, weight_decay=0.0,
    device='cpu', early_stopping=False, restore_best=False, monitor_train=True)
N_TREE_SHADOWS = 16
SHADOW_PLAN = tuple(PAPER_SHADOW_ALGORITHMS[index % len(PAPER_SHADOW_ALGORITHMS)]
    for index in range(N_TREE_SHADOWS)) + ('mlp',) * 4
assert N_SHADOW_MODELS == len(SHADOW_PLAN) == 20
assert {algorithm: SHADOW_PLAN.count(algorithm) for algorithm in SHADOW_ALGORITHMS} == {
    'random_forest': 6, 'gradient_boosting': 5, 'xgboost': 5, 'mlp': 4}
# Native commuting units; all other added codes are categorical in the walk.
EXPANDED_NUMERIC_WALK = {'JWMNP': (2.0, 0, 200), 'JWRIP': (1.0, 1, 10)}
print('Training size:', TRAIN_SIZE, '| Input fields:', len(EXPANDED_FEATURES))
print('Reference split configuration:', SPLIT_CONFIG)

import importlib
import src.notebook_output as notebook_output_module
importlib.reload(notebook_output_module)
notebook_output_module.suppress_sklearn_parallel_warning()
from src.notebook_output import CellProgress, quiet_output
assert len(EXPANDED_FEATURES) == 48 and 'VPS' in EXPANDED_FEATURES


## Load 2023 target records and make stratified partitions
Split sizes and seed follow Experiment 2, applied to the new 2023 records.


In [ ]:
expanded_source = load_expanded_year(2023, income_base_year=INCOME_BASE_YEAR)
reference_partitions = split_acs_data(expanded_source, **SPLIT_CONFIG)
expanded_base_partitions = reference_partitions
expanded_partitions = subset_target_training(
    expanded_base_partitions, train_size=TRAIN_SIZE, seed=SEED)
for name in ['validation', 'test', 'mia_members', 'mia_nonmembers']:
    np.testing.assert_array_equal(expanded_partitions[name].index, reference_partitions[name].index)
assert set(expanded_partitions['mia_members'].index).issubset(expanded_partitions['train'].index)
for name in ['validation', 'test', 'mia_nonmembers']:
    assert set(expanded_partitions[name].index).isdisjoint(expanded_partitions['train'].index)
print('Target income adjustment:', expanded_source.attrs.get('income_adjustment', NOMINAL_INCOME_METADATA))
print('Eligible records:', len(expanded_source))
print('Unused records:', len(expanded_source) - TRAIN_SIZE
      - sum(len(expanded_partitions[name]) for name in ['validation', 'test', 'mia_nonmembers']))
display(pd.DataFrame({name: {'records': len(part), 'positive_rate': part['income_gt_50k'].mean()}
    for name, part in expanded_partitions.items()}).T)
feature_inventory = expanded_feature_inventory(expanded_partitions['train'])
display(feature_inventory)


## Preprocess using training records only
The output is sparse; the MLP densifies only the current batch. Cardinality
counts above refer to the original fields, not one-hot dimensions.


In [ ]:
expanded_preprocessor, expanded_arrays = preprocess_expanded_partitions(expanded_partitions)
print('Encoded input dimensions:', expanded_arrays['train']['X'].shape[1])
print('Training CSR storage (MiB):', round(sum(
    value.nbytes for value in [expanded_arrays['train']['X'].data,
    expanded_arrays['train']['X'].indices, expanded_arrays['train']['X'].indptr]) / 2**20, 2))
pd.DataFrame({name: {'rows': values['X'].shape[0], 'encoded_features': values['X'].shape[1]}
    for name, values in expanded_arrays.items()}).T


## Train the intermediate target near a 10-point validation accuracy gap
After every epoch, evaluate the current model on training and validation rows.
Define the gap as 100 * (train accuracy - validation accuracy), in percentage
points. Stop at the first epoch with gap >= 10, or after 30 epochs at most.
Keep the weights of the closest observed epoch to 10 points; ties keep the earlier
epoch. Thus a jump from 9 to 14 points returns the 9-point epoch. If the target
gap is never reached, return the closest visited epoch and report that limitation.

Only validation and training data determine this selection. Test and membership
metrics are evaluated afterward. A near-10-point validation gap need not produce
an identical test gap. This replaces fixed 30-epoch stopping, not the data split,
architecture, optimizer, shadow schedule or attack protocol. Weights are stored
temporarily in memory; no disk checkpoints or logs are used.

Restart the kernel and run the notebook in order when the experiment is authorized.
Existing outputs, if present, belong to the previous fixed-30-epoch configuration
and must not be interpreted as results of this updated selection rule.


In [ ]:
target_started = time.monotonic()
with quiet_output():
    expanded_model, expanded_history, expanded_metadata = train_target_model(
        expanded_arrays['train']['X'], expanded_arrays['train']['y'],
        expanded_arrays['validation']['X'], expanded_arrays['validation']['y'], **TRAIN_CONFIG)
target_training_seconds = time.monotonic() - target_started
print('Target training seconds:', round(target_training_seconds, 1))
print('Target regime:', TARGET_REGIME, '| Epochs run:', expanded_metadata['epochs_run'],
      '| Returned epoch:', expanded_metadata['returned_epoch'])
assert expanded_metadata['target_gap_pp'] == TRAIN_CONFIG['target_gap_pp']
assert 1 <= expanded_metadata['returned_epoch'] <= expanded_metadata['epochs_run'] <= TRAIN_CONFIG['epochs']
assert not expanded_metadata['early_stopping'] and not expanded_metadata['restore_best']
print('Requested train-validation gap (percentage points):', expanded_metadata['target_gap_pp'])
print('Selected train-validation gap (percentage points):',
      round(expanded_metadata['selected_train_validation_gap_pp'], 3))
print('Distance from requested gap (percentage points):', round(expanded_metadata['gap_distance_pp'], 3))
print('Stopping reason:', expanded_metadata['stopping_reason'])
if not expanded_metadata['gap_target_crossed']:
    print('The gap target was not reached within the epoch cap; returned the closest visited epoch.')
history_table = pd.DataFrame(expanded_history)
display(history_table.round(4))


## Income evaluation after validation-gap selection
Report train, validation and test accuracy and log loss for the selected model.
The training-validation accuracy gap was used to choose the epoch near 10 points.
Test data remain held out from selection, and the test gap can differ from 10.
These utility results are separate from the later membership-attack metrics.


In [ ]:
expanded_metrics = {name: evaluate_model(expanded_model, expanded_arrays[name]['X'],
    expanded_arrays[name]['y']) for name in ['train', 'validation', 'test']}
display(pd.DataFrame(expanded_metrics).T)
print('Train-test accuracy gap (percentage points):',
      100 * (expanded_metrics['train']['accuracy'] - expanded_metrics['test']['accuracy']))

print('Train-validation accuracy gap (percentage points):',
      100 * (expanded_metrics['train']['accuracy'] - expanded_metrics['validation']['accuracy']))
print('Validation minus train log-loss gap:',
      expanded_metrics['validation']['log_loss'] - expanded_metrics['train']['log_loss'])
print('Test minus train log-loss gap:',
      expanded_metrics['test']['log_loss'] - expanded_metrics['train']['log_loss'])

observed_validation_gap_pp = 100 * (
    expanded_metrics['train']['accuracy'] - expanded_metrics['validation']['accuracy'])
np.testing.assert_allclose(observed_validation_gap_pp,
    expanded_metadata['selected_train_validation_gap_pp'], rtol=0, atol=1e-8)


## Fixed reference pool: 16 trees plus four 100-epoch MLP shadows
Use exactly the reference's six RF, five gradient boosting and five XGBoost
shadows in slots 1–16, then four MLPs in slots 17–20. Settings, randomized
40/20/40 splits and seeds remain fixed (42 + 10 * index). Each preprocessor
fits its shadow training records only. Shadow OUT is not used for selection.

The MLP shadows use (512, 256, 128), dropout 0, Adam learning rate 0.001,
weight decay 0, batch size 256, CPU and exactly 100 epochs, final weights.
Their seeds are 202, 212, 222 and 232. SHADOW_MLP_CONFIG is deliberately
independent of TRAIN_CONFIG: shadow early stopping is off in both new variants.
This isolates target stopping under a fixed reference attack. It is architecture
matched but does not match the early-stopped or validation-gap-selected target training schedule;
a weak attack therefore does not establish absence of membership leakage.

All shadows use eligible 2014 records, disjoint from the 2023 target records.
Each shadow trains on about 73,576 rows; the target trains on 100,000 rows.
The same 48-field preprocessing procedure is fitted separately per model, so
one-hot widths can differ. Records may recur across shadows; attack record-ID
grouping keeps them together. The classifier and auxiliary calibration are
rebuilt with the same settings and seeds, without any target-label tuning.


In [ ]:
acs_auxiliary = load_expanded_year(2014, income_base_year=INCOME_BASE_YEAR)
print('Auxiliary income adjustment:', acs_auxiliary.attrs.get('income_adjustment', NOMINAL_INCOME_METADATA))
assert set(acs_auxiliary.index).isdisjoint(expanded_source.index)
shadow_results = {}
shadow_progress = CellProgress()
for index, algorithm in enumerate(SHADOW_PLAN):
    name = f'{algorithm}_{index + 1:02d}'
    shadow_progress.update(f'Training shadow {index + 1}/{N_SHADOW_MODELS}: {name}')
    with quiet_output():
        result = train_shadow_model(acs_auxiliary, algorithm, seed=SEED + 10 * index, schema='expanded_acs',
            parameters=SHADOW_MLP_CONFIG if algorithm == 'mlp' else None)
    result['attack_data']['shadow_model'] = name
    shadow_results[name] = result
shadow_progress.update(f'Completed {len(shadow_results)} shadows')
assert len(shadow_results) == N_SHADOW_MODELS == 20
assert sum(result['algorithm'] == 'mlp' for result in shadow_results.values()) == 4
for name, result in shadow_results.items():
    if result['algorithm'] == 'mlp':
        assert result['metadata']['epochs_run'] == SHADOW_MLP_CONFIG['epochs']
        assert result['metadata']['returned_epoch'] == SHADOW_MLP_CONFIG['epochs']
        assert result['metadata']['hidden_dims'] == list(SHADOW_MLP_CONFIG['hidden_dims'])
        assert result['metadata']['seed'] == result['seed']
# Verify native-input queries reproduce probabilities computed during training.
for name, result in shadow_results.items():
    audit_rows = result['attack_data'].iloc[:32]
    native = acs_auxiliary.loc[audit_rows['record_id'], EXPANDED_FEATURES]
    query = make_probability_oracle(result['model'], result['preprocessor'], schema='acs')
    np.testing.assert_allclose(query(native),
        audit_rows[['p_income_le_50k', 'p_income_gt_50k']].to_numpy(),
        rtol=1e-5, atol=1e-6, err_msg=f'Query mismatch for {name}')
print('Probability oracle consistency verified for all 20 shadows.')
display(summarize_shadow_models(shadow_results))


## Extract shadow features and fit the random forest attack
Features: class-scaled logit, blind walk, entropy and cross-entropy. Use 2,000
members and 2,000 non-members per shadow. Attack record-ID grouping prevents
repeated 2014 records from crossing attack partitions. Select both thresholds
on auxiliary validation only; no target membership labels train this classifier.


In [ ]:
with quiet_output():
    shadow_attack_data, shadow_query_diagnostics = collect_shadow_paper_features(
        shadow_results, acs_auxiliary, per_class=SHADOW_RECORDS_PER_CLASS,
        config=WALK_CONFIG, seed=SEED)
display(pd.DataFrame(shadow_query_diagnostics).T)
attack_result = train_membership_classifier(
    shadow_attack_data, seed=SEED, algorithm='random_forest', feature_set='paper')
display(pd.DataFrame(attack_result['metrics']).T)


## Final MIA evaluation on the expanded target
Query the expanded native input schema, preserving categorical codes and
handling numeric commute fields in native units. The walk still generates
field-valid but potentially implausible combinations, and its dimensionality
matches the expanded auxiliary schema. Thresholds are frozen from auxiliary validation.


In [ ]:
with quiet_output():
    target_attack_data, target_query_diagnostics = collect_target_paper_features(
        expanded_model, expanded_preprocessor, expanded_partitions,
        config=WALK_CONFIG, seed=SEED, numeric_protocol=EXPANDED_NUMERIC_WALK)
display(pd.Series(target_query_diagnostics, name='expanded target query diagnostics'))
target_mia_metrics, target_membership_scores = evaluate_attack(
    attack_result['classifier'], target_attack_data,
    threshold=attack_result['accuracy_threshold'],
    low_fpr_threshold=attack_result['low_fpr_threshold'])
target_mia_metrics['tpr_at_accuracy_threshold'] = target_mia_metrics['tp'] / target_mia_metrics['members']
target_mia_metrics['fpr_at_accuracy_threshold'] = target_mia_metrics['fp'] / (
    target_mia_metrics['records'] - target_mia_metrics['members'])
display(pd.DataFrame({'2014 auxiliary attack test': attack_result['metrics']['test'],
                      'Expanded ACS target MIA': target_mia_metrics}).T)


## Small reproducibility artifacts
Save summaries, the feature inventory and row assignments. Models remain in memory;
no checkpoints or log files are loaded or saved.


In [ ]:
result_dir = RUN_DIR
result_dir.mkdir(parents=True, exist_ok=True)
feature_inventory.to_csv(result_dir / 'feature_inventory.csv')
for name, part in expanded_partitions.items():
    pd.Series(part.index, name='record_id').to_csv(result_dir / f'{name}_ids.csv', index=False)
summarize_shadow_models(shadow_results).to_csv(result_dir / 'shadow_metrics.csv')
summary = dict(target_training_protocol=TARGET_TRAINING_PROTOCOL,
    income_base_year=INCOME_BASE_YEAR,
    target_income_adjustment=expanded_source.attrs.get('income_adjustment', NOMINAL_INCOME_METADATA),
    auxiliary_income_adjustment=acs_auxiliary.attrs.get('income_adjustment', NOMINAL_INCOME_METADATA),
    perturbation_protocol_audit={'shadow_numeric_fields': {'AGEP': [1.0, 17, 99], 'WKHP': [2.0, 1, 99]}, 'target_extra_numeric_fields': {'JWMNP': [2.0, 0, 200], 'JWRIP': [1.0, 1, 10]}, 'shadow_commuting_fields': 'Categorical replacements, unchanged from original run', 'target_commuting_fields': 'Numeric perturbations, unchanged from original run', 'correction': 'Prior assistant statements that the shadow commuting perturbation protocol changed were incorrect. Original and current perturbation call sources are identical.'},
    walk_numeric_protocol_scope='Extra target numeric fields only',
    walk_numeric_protocol=EXPANDED_NUMERIC_WALK, execution_status='acs_baseline_completed',
    attack_metrics_status='oracle_consistency_verified', target_year=2023, auxiliary_year=2014, n_shadow_models=N_SHADOW_MODELS,
    shadow_algorithms=list(SHADOW_ALGORITHMS),
    shadow_algorithm_plan=list(SHADOW_PLAN),
    shadow_counts={algorithm: SHADOW_PLAN.count(algorithm) for algorithm in SHADOW_ALGORITHMS},
    original_tree_shadow_count=N_TREE_SHADOWS, shadow_mlp_config=SHADOW_MLP_CONFIG,
    shadow_architecture_assumption='Target architecture known; fixed 100-epoch reference shadow schedule; target stopping differs; weights and training records unavailable',
    shadow_mlp_training={name: result['metadata'] for name, result in shadow_results.items()
        if result['algorithm'] == 'mlp'},
    shadow_seeds={name: result['seed'] for name, result in shadow_results.items()},
    original_feature_count=len(EXPANDED_FEATURES),
    encoded_feature_count=expanded_arrays['train']['X'].shape[1],
    train_size=TRAIN_SIZE, split_config=SPLIT_CONFIG, training=expanded_metadata,
    target_training_seconds=target_training_seconds, target_metrics=expanded_metrics,
    shadow_attack_metrics=attack_result['metrics'], target_mia_metrics=target_mia_metrics,
    target_query_diagnostics=target_query_diagnostics)
(result_dir / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Summary and exact row IDs written to', result_dir)


## Missing-feature PFV baseline
Use seven preselected masks: two singles, two pairs, two triples and one quadruple.
These retain the reduced K=6 study; the complete-feature baseline uses the historical settings.
All 48 target inputs remain required; the target is not retrained after masking.
Each hidden field is replaced before querying, including before blind walks.
The reconstruction pool contains ACS 2014 predictors only, with no income-label
conditioning. Naturally missing auxiliary values are excluded from the sampling pool.
Use stable-code fields only; do not extend this to occupation/industry without
an explicit cross-year category mapping. Related marriage and employment fields
remain observed. Income labels remain known for the original attack's loss features.

The same complete-feature shadow attack is used in every condition, isolating the
effect of target completion. There is no mask-specific attack retraining. Auxiliary
attack train/validation/test grouping and thresholds therefore remain unchanged.
A single seed is a baseline realization, not a measure of reconstruction variability.


In [ ]:
from itertools import combinations
from src.missing_feature_pfv import complete_pfv

MASK_FIELDS = ['MAR', 'WKHP', 'COW', 'SEX']
MISSING_MASKS = {
    'MAR': ['MAR'], 'WKHP': ['WKHP'],
    'MAR+COW': ['MAR', 'COW'], 'COW+SEX': ['COW', 'SEX'],
    'MAR+WKHP+SEX': ['MAR', 'WKHP', 'SEX'],
    'WKHP+COW+SEX': ['WKHP', 'COW', 'SEX'],
    'MAR+WKHP+COW+SEX': ['MAR', 'WKHP', 'COW', 'SEX'],
}
MISSING_DIR = RUN_DIR / 'missing_feature_pfv'
MISSING_DIR.mkdir(parents=True, exist_ok=True)
missing_rows = [{'condition': 'complete', 'n_missing': 0, **target_mia_metrics}]
missing_audits = {}
for mask_name, hidden in MISSING_MASKS.items():
    completed_partitions = {}
    # One shared completion pass: membership never selects a seed or distribution.
    candidates = pd.concat([expanded_partitions[name] for name in
        ['mia_members', 'mia_nonmembers']]).sort_index()
    completed_all = complete_pfv(candidates[EXPANDED_FEATURES],
        acs_auxiliary[EXPANDED_FEATURES], hidden, seed=SEED)
    completed_all['income_gt_50k'] = candidates['income_gt_50k']
    for name in ['mia_members', 'mia_nonmembers']:
        original = expanded_partitions[name]
        completed_partitions[name] = completed_all.loc[original.index].copy()
    with quiet_output():
        features, audit = collect_target_paper_features(
            expanded_model, expanded_preprocessor, completed_partitions,
            config=WALK_CONFIG, seed=SEED, numeric_protocol=EXPANDED_NUMERIC_WALK)
    metrics, scores = evaluate_attack(attack_result['classifier'], features,
        threshold=attack_result['accuracy_threshold'],
        low_fpr_threshold=attack_result['low_fpr_threshold'])
    metrics['tpr_at_accuracy_threshold'] = metrics['tp'] / metrics['members']
    metrics['fpr_at_accuracy_threshold'] = metrics['fp'] / (metrics['records'] - metrics['members'])
    missing_rows.append({'condition': mask_name, 'n_missing': len(hidden), **metrics})
    missing_audits[mask_name] = audit
    features.assign(membership_score=scores).to_csv(MISSING_DIR / f'{mask_name}_scores.csv', index=False)
missing_comparison = pd.DataFrame(missing_rows).set_index('condition')
missing_comparison['auc_change_from_complete'] = missing_comparison['roc_auc'] - target_mia_metrics['roc_auc']
display(missing_comparison)
missing_comparison.to_csv(MISSING_DIR / 'comparison.csv')
(MISSING_DIR / 'protocol.json').write_text(json.dumps(dict(
    method='PFV_single_draw', auxiliary_year=2014, target_year=2023,
    target_training_protocol=TARGET_TRAINING_PROTOCOL,
    masks=MISSING_MASKS, seed=SEED,
    naturally_missing_auxiliary_values='excluded',
    classifier='frozen complete-feature ACS 2014 attack',
    thresholds='frozen auxiliary validation', diagnostics=missing_audits,
    baseline_summary='../summary.json'), indent=2), encoding='utf-8')


## Uncertainty-aware MF-MIA: study protocol
Research question: does reconstruction-induced score standard deviation improve
membership inference beyond the mean? The following cells are additional experiments;
they are not executed when this notebook is generated.

Use the seven preselected masks, target, 20 ACS 2014 shadows (16 trees plus four fixed-reference 100-epoch MLPs), four base
attack features and existing blind-walk protocol. The base attack is frozen.
Compare conditional single imputation, average score, learned mean-only RF and
matched mean+standard-deviation RF on identical completion draws. PFV above is a
separate reconstruction baseline, not the single-imputation control for this section.

Sequential forest-leaf donor sampling conditions each hidden field on visible
fields and earlier sampled hidden fields (MAR, WKHP, COW, SEX order). It preserves
observed auxiliary values but is an approximate predictive distribution, not a
calibrated posterior. Reconstruction training excludes all base-attack, meta-fit,
calibration and auxiliary-test candidate IDs. No income label conditions imputation.
Auxiliary validation IDs are divided into meta-fit and calibration groups. Auxiliary
test IDs remain untouched until evaluation. The base attack's earlier validation
thresholds are not reused for this comparison. No target labels select methods,
masks, K or thresholds. Income class labels remain known to the base loss feature.

K=6 and one reconstruction seed are predeclared exploratory settings, not a
confirmatory multi-seed study. All seven masks are reported without choosing a winner
on target results. Raw draws remain in memory during this run.
The original commuting walk asymmetry is retained and limits interpretation.
Use two single-field, two double-field, two triple-field and one four-field mask;
each of MAR, WKHP, COW and SEX is hidden in four masks. Six draws give noisier
variability estimates than the original K=20 setting. The base attack is refitted
using all 20 shadows, with grouped auxiliary splits recomputed from
that pool. Reconstruction/meta/calibration roles follow those new splits.
Results use the separate exp_3_2023_2014_100000_nominal_tree16_mlp4_target_gap10_max30 artifact root
and reduced_k6_s20_m7 subdirectory. Previous tree-only and 16-shadow mixed results are not overwritten.
Common walk seeds reduce additional Monte Carlo noise; score spread can still
include completion-dependent effects of the finite blind-walk procedure.

This section can be expensive: each candidate is evaluated K times per mask,
with up to 41 record queries per completion under the original walk settings.
DP training/evaluation and repeated-seed confirmation are not implemented here.

Execution uses memory only for reconstruction models, audit draws, completion
scores and aggregators. No checkpoints are loaded or saved. Run these cells in
order in one kernel session; restarting requires computing them again.
Only final metrics, row assignments and the protocol are written to disk.

The target stopping rule is the only changed experimental factor relative to
the 100-epoch reference. Shadow models and auxiliary reconstruction/aggregation
remain fixed by recipe and seed. Compare the uncertainty versus learned_mean
contrast within each target condition before comparing contrasts across regimes.
One seed provides an exploratory comparison, not evidence of repeatability.


In [ ]:
# 1. Fixed configuration and disjoint auxiliary roles
import importlib
from sklearn.model_selection import train_test_split
from src.notebook_output import CellProgress, quiet_output
import src.uncertainty_mf_mia as uncertainty_module
# Pick up runtime-support edits without retraining/reloading the target or shadows.
importlib.reload(uncertainty_module)
from src.uncertainty_mf_mia import (
    ConditionalDonorChain, auxiliary_roles, completion_scores, summarize_scores,
    fit_aggregation, evaluate_aggregation,
)

UNCERTAINTY_K = 6
assert N_SHADOW_MODELS == len(shadow_results) == 20, 'Load the 20-shadow pool first.'
assert set(shadow_attack_data['shadow_model']) == set(shadow_results), 'Rebuild base attack data using all 20 shadows.'
assert set().union(*(set(part['shadow_model']) for part in attack_result['splits'].values())) == set(shadow_results), 'Refit the base attack on the 20-shadow pool.'
assert len(MISSING_MASKS) == 7, 'Use the seven preselected masks for the reduced study.'
RECONSTRUCTION_SEED = SEED + 1000
UNCERTAINTY_DIR = RUN_DIR / 'uncertainty_mf_mia'
UNCERTAINTY_DIR.mkdir(parents=True, exist_ok=True)
uncertainty_roles = auxiliary_roles(acs_auxiliary, attack_result, seed=SEED)
recon_fit_ids, recon_check_ids = train_test_split(
    sorted(uncertainty_roles['reconstruction']), test_size=0.2, random_state=SEED)
recon_training = acs_auxiliary.loc[recon_fit_ids, EXPANDED_FEATURES]
recon_check = acs_auxiliary.loc[recon_check_ids, EXPANDED_FEATURES]
display(pd.Series({name: len(ids) for name, ids in uncertainty_roles.items()}, name='unique_2014_records'))
uncertainty_reconstructors = {}


### Fit and audit conditional reconstruction using 2014 only
Held-out reconstruction checks use only reserved 2014 records, not attack candidates.
Numeric coverage is descriptive empirical coverage of the sampled 90% interval.
Categorical sample-match fraction measures draw accuracy; it is not calibrated
probability accuracy. Report these checks even when attack performance is poor.
Related fields remain visible, so reconstruction may be easy through proxies.


In [ ]:
# 2. Fit one conditional reconstruction chain per mask; no target queries
from src.notebook_output import CellProgress, quiet_output
reconstruction_audit_rows = []
reconstruction_progress = CellProgress()
for mask_index, (mask_name, hidden) in enumerate(MISSING_MASKS.items(), start=1):
    mask_started = time.monotonic()
    reconstruction_progress.update(f'[{mask_index}/{len(MISSING_MASKS)}] {mask_name}: fitting reconstruction')
    with quiet_output():
        reconstructor = ConditionalDonorChain(hidden, seed=RECONSTRUCTION_SEED).fit(recon_training)
    uncertainty_reconstructors[mask_name] = reconstructor
    reconstruction_progress.update(f'[{mask_index}/{len(MISSING_MASKS)}] {mask_name}: auditing {UNCERTAINTY_K} completions')
    with quiet_output():
        samples = list(reconstructor.iter_samples(recon_check,
            seeds=[RECONSTRUCTION_SEED + draw for draw in range(UNCERTAINTY_K)]))
    for field in hidden:
        observed = recon_check[field].notna().to_numpy()
        if not observed.any():
            continue
        draws = np.column_stack([sample[field].to_numpy()[observed] for sample in samples])
        truth = recon_check[field].to_numpy()[observed]
        row = dict(mask=mask_name, field=field, records=int(observed.sum()))
        if field == 'WKHP':
            lower, upper = np.quantile(draws, [0.05, 0.95], axis=1)
            row.update(mae_of_sample_mean=float(np.abs(draws.mean(axis=1) - truth).mean()),
                sampled_90pct_coverage=float(((truth >= lower) & (truth <= upper)).mean()),
                mean_interval_width=float((upper - lower).mean()))
        else:
            row['sample_match_fraction'] = float((draws == truth[:, None]).mean())
        reconstruction_audit_rows.append(row)
    reconstruction_progress.update(f'[{mask_index}/{len(MISSING_MASKS)}] {mask_name}: complete ({time.monotonic() - mask_started:.1f}s)')
reconstruction_audit = pd.DataFrame(reconstruction_audit_rows)
display(reconstruction_audit.round(4))
reconstruction_audit.to_csv(UNCERTAINTY_DIR / 'reconstruction_audit.csv', index=False)


### Generate held-out auxiliary completion scores
The original 2,000 member/nonmember samples per shadow are reused. Only meta-fit,
calibration and auxiliary-test records are queried here; base-attack training
records are excluded. Repeated IDs retain the same role across all shadows.
Completion never receives membership labels as a conditioning variable.


In [ ]:
# 3. K scores per held-out shadow observation, separately for every mask
from src.notebook_output import CellProgress, quiet_output
uncertainty_auxiliary_tables = {}
uncertainty_query_audits = {}
auxiliary_progress = CellProgress()
heldout_ids = set().union(*(uncertainty_roles[role] for role in ['meta', 'calibration', 'test']))
for mask_name, reconstructor in uncertainty_reconstructors.items():
    mask_tables = []
    for shadow_name, shadow in shadow_results.items():
        selected = shadow_attack_data.loc[
            (shadow_attack_data['shadow_model'] == shadow_name)
            & shadow_attack_data['record_id'].isin(heldout_ids),
            ['record_id', 'membership']].sort_values('record_id').reset_index(drop=True)
        if selected.empty:
            continue
        assert selected['record_id'].is_unique
        candidates = acs_auxiliary.loc[selected['record_id']]
        oracle = make_probability_oracle(shadow['model'], shadow['preprocessor'], schema='acs')
        auxiliary_progress.update(f'Auxiliary scoring: {mask_name} / {shadow_name}, {len(selected):,} records, K={UNCERTAINTY_K}')
        with quiet_output():
            draw_scores, audit = completion_scores(oracle, candidates, reconstructor,
                attack_result['classifier'], k=UNCERTAINTY_K, seed=RECONSTRUCTION_SEED,
                walk_config=WALK_CONFIG)  # preserve existing shadow walk protocol
        table = summarize_scores(draw_scores)
        table['record_id'] = selected['record_id'].to_numpy()
        table['membership'] = selected['membership'].to_numpy()
        table['shadow_model'] = shadow_name
        table['role'] = [next(role for role in ['meta', 'calibration', 'test']
            if record_id in uncertainty_roles[role]) for record_id in table['record_id']]
        mask_tables.append(table)
        uncertainty_query_audits[f'{mask_name}/{shadow_name}'] = audit
    uncertainty_auxiliary_tables[mask_name] = pd.concat(mask_tables, ignore_index=True)
auxiliary_progress.update(f'Auxiliary scoring complete: {len(uncertainty_auxiliary_tables)} masks')


In [ ]:
# 4. Fit aggregators, freeze thresholds on calibration, evaluate auxiliary test
from src.notebook_output import quiet_output
uncertainty_aggregators = {}
uncertainty_auxiliary_result_rows = []
for mask_name, table in uncertainty_auxiliary_tables.items():
    meta = table.loc[table['role'] == 'meta']
    calibration = table.loc[table['role'] == 'calibration']
    auxiliary_test = table.loc[table['role'] == 'test']
    with quiet_output():
        learned, thresholds = fit_aggregation(meta, calibration, seed=SEED)
    uncertainty_aggregators[mask_name] = (learned, thresholds)
    with quiet_output():
        metrics, predictions = evaluate_aggregation(auxiliary_test, learned, thresholds)
    uncertainty_auxiliary_result_rows.extend(metrics.assign(mask=mask_name,
        n_missing=len(MISSING_MASKS[mask_name]), dataset='2014_auxiliary_test').to_dict('records'))
print('Auxiliary test: macro averages across masks of the same size')
display(pd.DataFrame(uncertainty_auxiliary_result_rows).groupby(['n_missing', 'method'])[
    ['roc_auc', 'balanced_accuracy', 'tpr_at_1pct_fpr_roc']].mean().round(4))


### Final target evaluation: frozen methods and thresholds
Use the same target member/nonmember IDs as the complete-feature baseline.
Membership labels are attached only after scores are generated. The same K draws
feed single, averaging and both learned aggregators. Do not tune on these results.
The complete-feature and PFV results above are context; the controlled test for
the standard-deviation signal compares uncertainty against learned_mean here.


In [ ]:
# 5. Query the existing ACS 2023 target; no target retraining
from src.notebook_output import CellProgress, quiet_output
uncertainty_result_rows = list(uncertainty_auxiliary_result_rows)
target_candidates = pd.concat([expanded_partitions[name]
    for name in ['mia_members', 'mia_nonmembers']]).sort_index()
target_member_ids = set(expanded_partitions['mia_members'].index)
target_oracle = make_probability_oracle(expanded_model, expanded_preprocessor, schema='acs')
target_progress = CellProgress()
for mask_name, reconstructor in uncertainty_reconstructors.items():
    target_progress.update(f'Target scoring: {mask_name}, K={UNCERTAINTY_K}')
    with quiet_output():
        draw_scores, audit = completion_scores(target_oracle, target_candidates,
            reconstructor, attack_result['classifier'], k=UNCERTAINTY_K,
            seed=RECONSTRUCTION_SEED, walk_config=WALK_CONFIG,
            numeric_protocol=EXPANDED_NUMERIC_WALK)
    table = summarize_scores(draw_scores)
    table['record_id'] = target_candidates.index.to_numpy()
    table['membership'] = table['record_id'].isin(target_member_ids).astype(int)
    learned, thresholds = uncertainty_aggregators[mask_name]
    with quiet_output():
        metrics, predictions = evaluate_aggregation(table, learned, thresholds)
    uncertainty_result_rows.extend(metrics.assign(mask=mask_name,
        n_missing=len(MISSING_MASKS[mask_name]), dataset='2023_target').to_dict('records'))
    uncertainty_query_audits[f'{mask_name}/target'] = audit
uncertainty_results = pd.DataFrame(uncertainty_result_rows)
uncertainty_results.to_csv(UNCERTAINTY_DIR / 'comparison.csv', index=False)
target_progress.update(f'Target scoring complete: {len(uncertainty_reconstructors)} masks')
target_summary = uncertainty_results.loc[uncertainty_results['dataset'] == '2023_target']
print('Target: macro averages across masks of the same size')
display(target_summary.groupby(['n_missing', 'method'])[
    ['roc_auc', 'balanced_accuracy', 'tpr_at_1pct_fpr_roc']].mean().round(4))
print('Target ROC-AUC by mask; complete metrics are saved in comparison.csv')
display(target_summary.pivot(index='mask', columns='method', values='roc_auc').round(4))
_ = (UNCERTAINTY_DIR / 'protocol.json').write_text(json.dumps(dict(
    target_training_protocol=TARGET_TRAINING_PROTOCOL, target_training=expanded_metadata,
    k=UNCERTAINTY_K, reconstruction_seed=RECONSTRUCTION_SEED, masks=MISSING_MASKS,
    n_shadow_models=N_SHADOW_MODELS, shadow_names=list(shadow_results),
    shadow_algorithm_plan=list(SHADOW_PLAN), shadow_mlp_config=SHADOW_MLP_CONFIG,
    mask_selection='preselected coverage: 2 singles, 2 pairs, 2 triples, 1 quadruple; each field hidden four times',
    reconstruction='sequential random-forest leaf donor sampling; approximate predictive distribution',
    reconstruction_chain_order=MASK_FIELDS, score_std_ddof=0,
    auxiliary_year=2014, target_year=2023, target_labels_for_tuning=False,
    meta_source='half of base-attack validation record groups',
    calibration_source='other half of base-attack validation record groups',
    methods=['single', 'average', 'learned_mean', 'uncertainty'],
    queries=uncertainty_query_audits,
    limitations=['one reconstruction seed', 'no DP evaluation yet',
        'fixed 100-epoch shadow schedule differs from target stopping rule',
        'cross-year distribution shift', 'original blind-walk asymmetry retained',
        'no confirmatory significance claim; repeated shadow rows require grouped inference']),
    indent=2), encoding='utf-8')
